In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
from torch.utils.data import random_split, ConcatDataset, DataLoader
from torchvision.datasets import ImageFolder
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import numpy as np
import time
import os
from torchvision import datasets
import timm

/data/students/joshua_c/dev/ai231-group/.venv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Define transforms
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [8]:
model = timm.create_model('efficientnetv2_s', pretrained=False)
# Count total parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


Total parameters: 21,458,488
Trainable parameters: 21,458,488


In [3]:
# Load dataset using ImageFolder
train_dataset = datasets.ImageFolder(root="./brain_tumor_dataset/Training", transform=train_transform)
test_dataset = datasets.ImageFolder(root="./brain_tumor_dataset/Testing", transform=train_transform)
classes = test_dataset.classes
full_dataset = ConcatDataset([train_dataset, test_dataset])

train_dataset, test_dataset = random_split(
    full_dataset,
    [0.7, 0.3],
    generator=torch.Generator().manual_seed(42)
)

val_size = int(len(train_dataset) * 0.2)
train_size = len(train_dataset) - val_size
train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])
val_dataset.dataset.transform = val_transform
test_dataset.dataset.transform = val_transform

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [4]:
# === Load and Modify EfficientNet-B0 ===
model = timm.create_model('efficientnetv2_s', pretrained=True)

# Unfreeze all layers for full fine-tuning
for param in model.parameters():
    param.requires_grad = True

# Add dropout and replace classifier
num_classes = 4
model.classifier = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(model.classifier[1].in_features, num_classes)
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

RuntimeError: No pretrained weights exist for efficientnetv2_s. Use `pretrained=False` for random init.

In [ ]:
# Optimizer, Scheduler, Loss Function
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-5, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=4, verbose=True)

In [ ]:
# Early stopping setup
class EarlyStopping:
    def __init__(self, patience=6):
        self.patience = patience
        self.best_score = None
        self.counter = 0
        self.best_model = None

    def step(self, score, model):
        if self.best_score is None or score > self.best_score:
            self.best_score = score
            self.counter = 0
            self.best_model = model.state_dict()
        else:
            self.counter += 1
        return self.counter >= self.patience

# Training function with early stopping
def train_model(model, train_loader, valid_loader, epochs=40):
    early_stopping = EarlyStopping(patience=6)

    for epoch in range(epochs):
        model.train()
        train_loss, correct, total = 0.0, 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_acc = correct / total
        val_acc = evaluate(model, valid_loader)
        scheduler.step(val_acc)

        print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss/len(train_loader.dataset):.4f}, "
              f"Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")

        if early_stopping.step(val_acc, model):
            print("Early stopping triggered.")
            break

    model.load_state_dict(early_stopping.best_model)

# Evaluation function
def evaluate(model, dataloader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

# Train the model
train_model(model, train_loader, valid_loader)

# Final test set evaluation
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        y_true.extend(labels.numpy())
        y_pred.extend(preds.cpu().numpy())

print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=classes))
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Epoch 1/40, Train Loss: 1.3701, Train Acc: 0.3217, Val Acc: 0.4354
Epoch 2/40, Train Loss: 1.2493, Train Acc: 0.4781, Val Acc: 0.6258
Epoch 3/40, Train Loss: 1.1439, Train Acc: 0.5990, Val Acc: 0.7549
Epoch 4/40, Train Loss: 1.0417, Train Acc: 0.6674, Val Acc: 0.7637
Epoch 5/40, Train Loss: 0.9461, Train Acc: 0.7046, Val Acc: 0.7965
Epoch 6/40, Train Loss: 0.8566, Train Acc: 0.7429, Val Acc: 0.8118
Epoch 7/40, Train Loss: 0.7680, Train Acc: 0.7713, Val Acc: 0.8293
Epoch 8/40, Train Loss: 0.7138, Train Acc: 0.7845, Val Acc: 0.8403
Epoch 9/40, Train Loss: 0.6516, Train Acc: 0.7949, Val Acc: 0.8337
Epoch 10/40, Train Loss: 0.6061, Train Acc: 0.8080, Val Acc: 0.8621
Epoch 11/40, Train Loss: 0.5743, Train Acc: 0.8195, Val Acc: 0.8731
Epoch 12/40, Train Loss: 0.5396, Train Acc: 0.8255, Val Acc: 0.8775
Epoch 13/40, Train Loss: 0.5010, Train Acc: 0.8430, Val Acc: 0.8687
Epoch 14/40, Train Loss: 0.4717, Train Acc: 0.8441, Val Acc: 0.8862
Epoch 15/40, Train Loss: 0.4451, Train Acc: 0.8600, Val A

AttributeError: 'Subset' object has no attribute 'classes'

In [ ]:
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=classes))
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Classification Report:
                  precision    recall  f1-score   support

    glioma_tumor       0.94      0.94      0.94       123
meningioma_tumor       0.96      0.90      0.93       135
        no_tumor       0.96      1.00      0.98        71
 pituitary_tumor       0.96      0.99      0.98       128

        accuracy                           0.95       457
       macro avg       0.95      0.96      0.96       457
    weighted avg       0.95      0.95      0.95       457

Confusion Matrix:
[[116   5   2   0]
 [  7 122   1   5]
 [  0   0  71   0]
 [  1   0   0 127]]
